# Driven Square (2D)

A square rigid body **oscillates back and forth along x** -- `x(t) = amplitude *
cos(2*pi*t/period)`, starting from rest at `x = +amplitude` -- through an
otherwise still fluid. "Driven" names the body here, not the flow: the square is
what is being pushed, and the fluid starts at rest. The wake it sheds trails and
folds behind it each half-cycle, unlike a body translating in one direction
forever, so this is closer to a vortex-shedding oscillating-cylinder experiment
than a towing-tank run -- worth comparing against `13-openFlow.ipynb` (a fixed
obstacle, real channel walls, real inflow) and `10-moving-obstacle.ipynb` (a
body that spins in place instead of translating).

Three things follow from "back and forth" that would not matter for a body
moving in one direction forever:

- **The domain is sized for the excursion, not just the body.** The box is at
  least `domainMarginRatio` (default 2) times as wide as the body's total sweep
  -- `2 * (oscillationAmplitude + obstacleSize)`, since `obstacleSize` is a
  half-extent -- so the body never approaches the periodic wrap in either
  direction and has margin on both sides for its wake to develop before the next
  swing brings it back. `configureScheme` computes this explicitly rather than
  reusing the shared block's square box; the geometry-preview cell below prints
  the numbers.
- **The velocity is re-imposed every step, not set once.** `RigidBody.linearVelocity`
  is what the integrator actually reads each step to advance `centerOfMass`
  (`rigidBody/integrate.py`), so a one-shot assignment in `initialConditions`
  would leave the body translating in a straight line forever -- which is what
  the *first* version of this case did, before oscillation was asked for. A
  `postStep` hook (the same mechanism `kidder.py` uses to re-impose an analytic
  boundary velocity every step) recomputes `d/dt[A cos(2*pi*t/T)]` from the
  current `t` instead.
- **The body starts at rest, not at peak speed.** The first version of this
  oscillation had `x(t) = A sin(...)`, which is zero at `t = 0` but has
  *maximum* velocity there -- an instantaneous jump from the fluid's rest state
  to `amplitude * omega`, a real velocity discontinuity. Measured effect at
  this notebook's own `nx=128`, over the full 10s run: density excursion of
  -7.2%/+5.7% against rho0, against -4.1%/+3.5% once `x(t) = A cos(...)`
  starts the body at the `+A` extreme with zero velocity instead (`buildSystem`
  samples it there). For scale, `movingObstacle` -- unmodified, same
  resolution, same run length -- reaches -3.1%/+5.0% on its own: some density
  excursion past the usual +-1% is a shared characteristic of a rigid body
  driven through a small, periodic, nearly inviscid box (there is no outflow
  to carry acoustic energy away, and it worsens with resolution because
  `alpha`'s artificial viscosity scales with `h`, which shrinks as `nx`
  grows), not something either case fully avoids -- but the instantaneous
  velocity jump was a second, avoidable problem on top of it, and removing it
  is most of the gap between -7.2% and -4.1%.

One thing is deliberately *off* by default: **no freestream**.
`--enableFreestream` layers `movingObstacle`'s mean-flow forcing (drives the
domain-*mean* velocity towards `U_target`, leaving fluctuations alone) under the
oscillation, so the body swings through a driven current instead of still
fluid -- a real, separate experiment, not what "driven square" asks for by
default. There is also no `--bounded`/`--band` channel-confinement flag:
confining an oscillating body with real walls is already the domain-sizing
problem above solved a different, more restrictive way, and `13-openFlow.ipynb`
already covers a *fixed* obstacle inside real channel walls if that's what's
wanted.

![](outputs/11-drivenSquare.gif)

## Every knob, and what it does

The parameters cell below is the whole command line of `11-driven-square.py`
written out: `CaseSpec` fields first, then `drivenSquareCase.params` -- the
case's own physics knobs, each of which is also a `--flag`. Anything not named
there keeps the value in `drivenSquareCase.defaults`/`.params`.

**Discretisation, time stepping and output** (`CaseSpec` fields, shared by every case)

| field | this notebook | what it does |
|---|---|---|
| `nx` | `128` | particles across the domain's *y* extent (`L`); the spacing is `dx = L / nx` and applies to x too |
| `dim` | `2` | this case is 2D |
| `L` | `2.0` | the domain's y extent; the x extent is computed from the oscillation, below |
| `n_h` | `4.0` | particles per support radius, i.e. how smooth the kernel is |
| `kernel` | `Wendland4` | SPH kernel |
| `integrationScheme` | `rungeKutta2` | time integrator |
| `scheme` | `deltaSPH` | the solver itself |
| `tLimit` | `10.0` | simulated end time; the loop runs `tLimit / dt` steps -- 2.5 periods at the defaults below |
| `dt` | *set by the case* | left `None`: `initialConditions` picks it together with the sound speed |
| `adaptiveDt`, `cflFactor`, `minDt` | `True`, `0.3`, `1e-8` | CFL limiter around that `dt` |
| `plot`, `show`, `plotInterval` | `True`, `True`, `10` | render a frame every `plotInterval` steps |
| `store`, `storeMode`, `storeInterval` | `False`, `'states'`, `500` | HDF5 export; off here |

**The case's own parameters** (`--flag` on the script, `params=dict(...)` here)

| parameter | this notebook | what it does |
|---|---|---|
| `obstacleShape` | `'box'` | any key of `SHAPE_PRESETS`; `box` at `obstacleAspect=1` is the square this case is named for |
| `obstacleSize` | `0.25` | characteristic **half**-extent of the body -- what the domain sizing below adds on each side of the swing |
| `obstacleAspect` | `1.0` | squashes it in its second direction; `1.0` keeps it square |
| `obstacleRotation` | `0.0` | degrees counter-clockwise, its initial orientation |
| `obstacleOffset` | `[0.0, 0.0]` | added to where the oscillation itself places the body (`x = +oscillationAmplitude`, `y = 0`) -- use this for a vertical shift or an extra x bias, not to set the starting x directly; a list, so `--config`/notebook only |
| `oscillationAmplitude` | `0.5` | half the peak-to-peak swing, in x -- also where the body starts (at rest) |
| `oscillationPeriod` | `4.0` | time for one full back-and-forth cycle |
| `domainMarginRatio` | `2.0` | the domain's x extent is at least this many times `2 * (oscillationAmplitude + obstacleSize)` |
| `enableFreestream` | `False` | layer mean-flow forcing under the oscillation -- swinging through a current rather than still fluid |
| `U_target` | `1.0` | the mean x-velocity the forcing drives the fluid towards, when `enableFreestream` is set |
| `forcingTau` | `0.5` | timescale of that forcing; larger is gentler |
| `rho0` | `1.0` | rest density |
| `targetDt` | `0.0005` | the timestep the run *asks* for; the sound speed is then chosen to make it the acoustic CFL limit |
| `inviscid`, `nu` | `True`, `0.0` | physical viscosity: `inviscid=True` leaves the scheme's own dissipation as the only one |
| `alpha` | `0.01` | artificial-viscosity coefficient; see `05-taylor-green-vortex.ipynb` for what it means and its stability floor |
| `freeSurface` | `False` | surface detection; this case has no free surface |
| `band` | `0` | particle layers of boundary padding; none, the box is periodic |
| `markerSize` | `8` | plot only: particle marker size |

**Three things this family does differently from the compressible notebooks**
(they will bite if `../compressible/08-Hydrostatic.ipynb` is copied unread):

1. The IC cell has a **fourth call**, `drivenSquareCase.initialConditions(ctx, system)`.
   That is where the body's starting velocity (zero -- see above) and the
   optional freestream forcing are installed, and where
   `setupWeaklyCompressibleTimestep` picks the sound speed and `config.dt`
   *together* from `targetDt`. Skip it and `config.dt` stays `None`. (The
   body's *position* is set earlier, in `buildSystem`, not here.)
2. **The loop is `range(nSteps)`.** No case in this family defines a `timestep`
   hook, so `dt` is fixed for the whole run after step 1 and `while t < tLimit`
   would be the wrong shape. This case *does* have a `postStep` hook -- but that
   re-imposes the oscillation velocity, it does not change `dt`.
3. Plotting calls `buildFieldPlotter`/`refreshFieldPlotter` on `VELOCITY_DENSITY_FIELDS`
   directly rather than `drivenSquareCase.setupPlot`/`updatePlot`, which go
   through `openWindow`/`pumpEvents` and do not live-update inside a Jupyter
   cell in this environment -- `08-Hydrostatic.ipynb` explains that in full.

Precision note: switching between single and double precision is controlled in
the import cell below. Because precision is set when core modules/kernels are
initialized, any precision change requires a kernel restart.

In [ ]:
%matplotlib inline
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.drivenSquare import drivenSquareCase, oscillationVelocity, sweptWidth
from warpSPH.cases.plotting import buildFieldPlotter, refreshFieldPlotter
from warpSPH.cases.weaklyCompressible import VELOCITY_DENSITY_FIELDS
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `11-driven-square.py`,
# made explicit and editable here -- the table in the intro cell says what each
# one does. `drivenSquareCase.defaults`/`.params` are the same values the CLI
# script starts from.
spec = CaseSpec(caseName=drivenSquareCase.name, scheme=drivenSquareCase.scheme,
                params=dict(drivenSquareCase.params)) \
    .merged(**drivenSquareCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=128,
    dim=2,
    L=2.0,

    # --- time stepping ---------------------------------------------------
    tLimit=10.0,

    # --- output --------------------------------------------------------------
    caseName='11-drivenSquare',
    plot=True, show=True, plotInterval=10,
    store=False,

    # --- the body's own knobs ---------------------------------------------
    params=dict(
        # the body: a square by default, any shape from SHAPE_PRESETS
        obstacleShape='box', obstacleSize=0.25, obstacleAspect=1.0,
        obstacleRotation=0.0, obstacleOffset=[0.0, 0.0],
        # how it moves: back and forth along x
        oscillationAmplitude=0.5, oscillationPeriod=4.0,
        # the domain's x extent is at least this many times the body's total
        # excursion -- see the intro cell for the formula
        domainMarginRatio=2.0,
        # off by default -- see the intro cell for why
        enableFreestream=False, U_target=1.0, forcingTau=0.5,
        # the fluid
        rho0=1.0, targetDt=0.0005, inviscid=True, nu=0.0,
        freeSurface=False, band=0,
        markerSize=8,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`drivenSquareCase.configureScheme`/`.buildSystem`), not re-derived here.
#
# `configureScheme` is where the domain's x extent is widened for the swing --
# every other case in this family leaves the shared block's square box alone.
# `buildSystem` is where the body is sampled at x = +oscillationAmplitude
# (rest, not the domain centre at peak speed -- see the intro cell for why
# that matters). `initialConditions` is the call the compressible notebooks do
# not have: the body's starting velocity (zero) and the optional freestream
# forcing are installed there, and it is where the sound speed and `config.dt`
# are chosen together from `targetDt`, so skipping it leaves `config.dt` unset.
ctx = buildContext(drivenSquareCase, spec)
drivenSquareCase.configureScheme(ctx)
system = drivenSquareCase.buildSystem(ctx)
drivenSquareCase.initialConditions(ctx, system)
runningState = system.initializeNewState()

domain = ctx.config.domain
swept = sweptWidth(ctx)
width = float(domain.max[0] - domain.min[0])
body = ctx.config.rigidBodies[0]
print(f'dt = {float(ctx.config.dt):.3e}, '
      f'c0 = {ctx.schemeConfig.fluid.fixedSoundSpeed:.3f}, '
      f'{len(runningState.state.positions)} particles')
print(f'domain: x in [{domain.min[0]:.3g}, {domain.max[0]:.3g}] '
      f'(width {width:.3g}), y in [{domain.min[1]:.3g}, {domain.max[1]:.3g}]')
print(f'body sweep: {swept:.3g} (domain width / sweep = {width / swept:.2g}, '
      f"margin on each side: {(width - swept) / 2:.3g})")
print(f'starting position: {body.centerOfMass.tolist()}, '
      f'starting velocity: {body.linearVelocity.tolist()} (should be ~0)')
print(f'x(t) = {spec.param("oscillationAmplitude")} * '
      f'cos(2*pi*t / {spec.param("oscillationPeriod")})')

In [ ]:
# What was actually built: the sampled regions, fluid and the body, against the
# domain (black) the run is periodic in. This is the cell to look at when a
# shape parameter above did something other than what it sounded like, and it
# is where the domain-sizing formula above turns into a picture: the body's
# full sweep with `domainMarginRatio`'s worth of clearance visible on both
# sides at x = 0 -- the body itself starts at the right-hand dotted line (its
# rest position, x = +amplitude), not at the centre.
figure, axis = plt.subplots(1, 1, figsize=(8, 4), squeeze=False)
plotRegions(ctx.scratch['regions'], axis[0, 0], plotFluid=True, plotParticles=True)
amplitude = spec.param('oscillationAmplitude')
size = spec.param('obstacleSize')
for x in (-amplitude - size, amplitude + size):
    axis[0, 0].axvline(x, color='red', ls=':', lw=1, alpha=0.6)
axis[0, 0].set_aspect('equal')
axis[0, 0].set_xlim(domain.min[0].item(), domain.max[0].item())
axis[0, 0].set_ylim(domain.min[1].item(), domain.max[1].item())
axis[0, 0].set_title(f'{len(ctx.scratch["regions"])} regions, '
                     f'{len(runningState.state.positions)} particles '
                     '(dotted: extent of the swing)')
figure.tight_layout()

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct buildFieldPlotter(VELOCITY_DENSITY_FIELDS), not
# drivenSquareCase.setupPlot -- see the intro cell for why.
plotter = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    plotter = buildFieldPlotter(ctx, runningState, VELOCITY_DENSITY_FIELDS,
                                figsize=(width / max(1.0, spec.L) * 5, 5))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = drivenSquareCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData,
                              extraFields=drivenSquareCase.extraFields)

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it --
# `drivenSquareCase.postStep(ctx, runningState, i)` right after the integrator
# call is what re-imposes the oscillation velocity every step; drop it and the
# body would coast off in a straight line at whatever velocity it started with.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

# The body's own centreOfMass is not part of `weaklyCompressibleDiagnostics`
# (nothing else in this family has a moving rigid body), so it's injected at
# the hook point rather than added there for one case.
trajectory = [dict(drivenSquareCase.diagnostics(ctx, runningState), step=-1, t=0.0)]
bodyPosition = [ctx.config.rigidBodies[0].centerOfMass.tolist()]
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    drivenSquareCase.postStep(ctx, runningState, i)
    # -------------------------------------------------------------------------

    bodyPosition.append(ctx.config.rigidBodies[0].centerOfMass.tolist())
    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = drivenSquareCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if plotter is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        refreshFieldPlotter(ctx, runningState, plotter, VELOCITY_DENSITY_FIELDS, step=i)

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                   schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                   extraFields=drivenSquareCase.extraFields)

In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## Did the body oscillate the way it was told to, and did it stay clear of the walls?

`centerOfMass` is what `postStep` drives every step, so the left panel checks
that it actually tracks `x(t) = amplitude * cos(2*pi*t/T)` (a small drift from
the analytic curve is expected -- the velocity is exact but `centerOfMass` is
still Euler-integrated from it one `dt` at a time, the same accuracy tradeoff
`kidder.py` accepts for the same reason) and the red band is the domain's
margin either side of the swing: the whole point of the `domainMarginRatio`
sizing was for the curve to never come close to it.

The right-hand panel is the standing weakly compressible health check: the
density bounds against the +-1% band the scheme's whole premise rests on. It
is also the check that caught the velocity-discontinuity problem the intro
cell describes -- the `A sin(...)` phase blew past it substantially more than
this notebook does now, which is what motivated starting the body at rest
instead. Some excursion past the band is expected here regardless (see the
intro cell -- it is a shared characteristic of this small-periodic-box,
low-viscosity family, not unique to this case), so the bar for this panel is
"comparable to `movingObstacle`'s own", not "flat".

In [ ]:
figure, axis = plt.subplots(1, 2, figsize=(11, 4))
t = np.array([row['t'] for row in trajectory])
position = np.array(bodyPosition)
expected = spec.param('oscillationAmplitude') * np.cos(2 * np.pi * t / spec.param('oscillationPeriod'))

axis[0].axhspan(swept / 2, width / 2, color='red', alpha=0.08, label='margin')
axis[0].axhspan(-width / 2, -swept / 2, color='red', alpha=0.08)
axis[0].plot(t, position[:, 0], label='centerOfMass x (measured)')
axis[0].plot(t, expected, ls=':', color='black', label='A cos(2*pi*t/T)')
axis[0].set_xlabel('t'); axis[0].set_ylabel('x'); axis[0].legend()
drift = float(np.abs(position[:, 0] - expected).max())
axis[0].set_title(f'max drift from analytic: {drift:.3g}')

axis[1].plot(t, [row['maxDensity'] for row in trajectory], label='max')
axis[1].plot(t, [row['minDensity'] for row in trajectory], label='min')
axis[1].axhspan(0.99, 1.01, color='green', alpha=0.1, label=r'$\pm 1\%$')
axis[1].set_xlabel('t'); axis[1].set_ylabel(r'$\rho$'); axis[1].legend()
figure.tight_layout()